In [1]:
import sys

from pathlib import Path
PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))

!{sys.executable} -m pip install scikit-surprise

In [2]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

In [3]:
from src.data.load_data import load_ratings
ratings = load_ratings()
print(ratings.shape)

(20000263, 4)


In [4]:
ratings_5m = ratings.sample(5_000_000, random_state=42)

*Dataset Sampling*   
 
We use a 5M sample of the MovieLens dataset for efficient experimentation and model comparison. All models are evaluated on the same subset to ensure fair comparison.

In [5]:
reader = Reader(rating_scale=(0.5, 5.0))

In [6]:
data = Dataset.load_from_df(ratings_5m[['userId', 'movieId', 'rating']], reader)

In [7]:
trainset, testset = train_test_split(data, test_size=0.2)

In [8]:
model = SVD()
model.fit(trainset)

In [9]:
predictions = model.test(testset)
rmse = accuracy.rmse(predictions)

RMSE: 0.8673


📌 Insight: Matrix factorization significantly improves performance
SVD achieves a substantially lower RMSE compared to baseline models.

Latent factors successfully capture hidden user preferences and item characteristics
The model learns interactions between users and items rather than treating them independently

In [10]:
ratings_1m = ratings.sample(1_000_000, random_state=42)

*Dataset Sampling (Hyperparameter Tuning)*

We further use a 1M sample of the MovieLens dataset to perform hyperparameter tuning. This reduced dataset enables efficient experimentation and significantly lowers computational cost while preserving the overall data distribution. The selected parameters are later validated on a larger dataset to ensure model robustness.

In [11]:
from surprise.model_selection import GridSearchCV

In [12]:
param_grid = {
    "n_factors": [50, 100, 150],
    "lr_all": [0.002, 0.005],
    "reg_all": [0.02, 0.05, 0.1]
}

In [13]:
gs = GridSearchCV(
    SVD,
    param_grid,
    measures=["rmse"],
    cv=3,
    n_jobs=1
)

In [14]:
gs.fit(data)
print(gs.best_score["rmse"])
print(gs.best_params["rmse"])

In [ ]:
model = SVD(
    n_factors=150,
    lr_all=0.005,
    reg_all=0.05
)

model.fit(trainset)
predictions = model.test(testset)
rmse = accuracy.rmse(predictions)

🎯 Hyperparameter tuning was performed on a 1M subset to efficiently identify promising configurations for SVD (n_factors, learning rate, regularization). The best parameters are subsequently evaluated on the full 5M dataset to verify their effectiveness in large-scale settings.

📌 Insight: the model achieved a slightly improved performance compared to the default setup:
Default SVD (5M): RMSE = 0.8673
Tuned SVD (5M, best params from 1M tuning): RMSE = 0.8637